### Example Exploratory Notebook

Use this notebook to explore the data generated by the pipeline in your preferred programming language.

**Note**: This notebook is not executed as part of the pipeline.

**Running via Databricks Connect (VS Code + serverless):** Python runs locally in `.venv`; only Spark / `dbutils` are remote proxies. So skip `%pip install` + `dbutils.library.restartPython()` — those are cluster-notebook idioms and fail on serverless with `cluster_id is required` (your uv venv also has no `pip`). `gtfs-realtime-bindings` is already a project dependency in `pyproject.toml`, so the import below resolves locally. To add more libraries: `uv add <pkg>` in a terminal.

In [6]:
%python

from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict
import pandas as pd

# Spark (serverless) reads the .pb bytes remotely; .collect() pulls them to local Python,
# where ParseFromString runs. Use binaryFile — dbutils.fs.head() corrupts binary data.
path = "abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/date=20260707/vline_tu_20260707T010000Z.pb"
raw = (spark.read.format("binaryFile")
       .load(path)
       .limit(1)
       .collect()[0]["content"])

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(raw)

print(f"{len(feed.entity)} entities; header ts = {feed.header.timestamp}")
print(feed.entity[0])  # human-readable dump of one record — shows the nested structure

59 entities; header ts = 1783385998
id: "01-SWL--6-T0-8080|20260707"
trip_update {
  trip {
    trip_id: "01-SWL--6-T0-8080"
    start_time: "06:52:00"
    start_date: "20260707"
    schedule_relationship: SCHEDULED
    route_id: "aus:vic:vic-01-SWL:"
  }
  stop_time_update {
    stop_sequence: 12
    arrival {
      delay: 282
      time: 1783384782
    }
    departure {
      delay: 408
      time: 1783384908
    }
    stop_id: "20315"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 13
    arrival {
      delay: 258
      time: 1783386498
    }
    stop_id: "22244"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 14
    arrival {
      delay: 258
      time: 1783387398
    }
    stop_id: "22240"
    schedule_relationship: SCHEDULED
  }
}



In [7]:
# Flatten to a DataFrame to explore the "columns".
# Note: GTFS-RT is nested — a TripUpdate has a `trip` message + a repeated `stop_time_update`.
# preserving_proto_field_name=True keeps snake_case field names (matches the GTFS-RT spec).
rows = [MessageToDict(e.trip_update, preserving_proto_field_name=True) for e in feed.entity]

# Top level: one row per trip
pd.json_normalize(rows).head()

# One row per stop (uncomment):
# pd.json_normalize(rows, record_path="stop_time_update", meta=[["trip", "trip_id"]]).head()

,stop_time_update,trip.trip_id,trip.start_time,trip.start_date,trip.schedule_relationship,trip.route_id
0,"[{'stop_sequence': 12, 'arrival': {'delay': 28...",01-SWL--6-T0-8080,06:52:00,20260707,SCHEDULED,aus:vic:vic-01-SWL:
1,"[{'stop_sequence': 19, 'arrival': {'delay': 12...",01-ECH--6-T0-8072,07:23:00,20260707,SCHEDULED,aus:vic:vic-01-ECH:
2,"[{'stop_sequence': 11, 'arrival': {'delay': 71...",01-WBL--6-T0-8861,07:26:00,20260707,SCHEDULED,aus:vic:vic-01-WBL:
3,"[{'stop_sequence': 12, 'arrival': {'time': '17...",01-SWL--6-T0-8081,07:38:00,20260707,SCHEDULED,aus:vic:vic-01-SWL:
4,"[{'stop_sequence': 21, 'arrival': {'delay': -1...",01-BDE--6-T0-8461,07:40:00,20260707,SCHEDULED,aus:vic:vic-01-BDE:
